In [1]:
# ==============================
# 1D-CNN for NLP (Kim CNN) - PyTorch
# 한국어 문장 분류 미니 예제 (자급자족)
# ==============================
import re, random, math
from typing import List, Tuple
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import pandas as pd

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

# -----------------------------
# 1) 데이터 (샘플)
#    label: 1(긍정), 0(부정)
# -----------------------------

DATA_PATH = "data/ratings_train.txt"

df = pd.read_csv(DATA_PATH, sep="\t")
df = df.head(1000)
raw_data = [
    ("오늘 카페라떼 진짜 맛있었다", 1),
    ("서비스가 너무 별로였어", 0),
    ("직원이 친절해서 기분이 좋았다", 1),
    ("커피가 미지근하고 맛없음", 0),
    ("분위기가 좋아서 또 오고 싶다", 1),
    ("가격이 너무 비싸고 양이 적다", 0),
    ("디저트가 전반적으로 훌륭했다", 1),
    ("테이블이 더러워서 실망했다", 0),
    ("음악 선곡이 좋고 편안했다", 1),
    ("주차가 너무 불편했다", 0),
]

# -----------------------------
# 2) 토크나이저
#    - 기본: 공백 기준 + 간단 정규화
#    - KOMORAN 사용 원하면 주석 해제
# -----------------------------
def normalize(s: str) -> str:
    s = re.sub(r"[^가-힣0-9a-zA-Z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# # KOMORAN 사용 버전 (선택)
from konlpy.tag import Komoran
komoran = Komoran()
KEEP = {"NNG","NNP","VV","VA","MAG","XR"}
STOP = {"하다","되다","이다"}
def komoran_tokenize(s: str) -> List[str]:
   s = normalize(s)
   return [m for (m,p) in komoran.pos(s) if p in KEEP and m not in STOP and len(m)>1]

tokenize = komoran_tokenize  # 변경 가능: tokenize = komoran_tokenize

texts, labels = df['document'].values, df['label'].values
tokens_list = [tokenize(t) for t in texts]


In [2]:

# -----------------------------
# 3) Vocab & 인코딩
# -----------------------------
min_count = 1
from collections import Counter
freq = Counter([w for toks in tokens_list for w in toks])
vocab = ["<PAD>", "<UNK>"] + [w for w,c in freq.items() if c >= min_count]
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

def encode(toks: List[str]) -> torch.Tensor:
    return torch.tensor([stoi.get(w, stoi["<UNK>"]) for w in toks], dtype=torch.long)

enc_inputs = [encode(toks) for toks in tokens_list]
labels_t = torch.tensor(labels, dtype=torch.long)

X_train, X_val, y_train, y_val = train_test_split(enc_inputs, labels_t, test_size=0.3, random_state=SEED, stratify=labels_t)


In [3]:

# -----------------------------
# 4) Dataset / DataLoader (패딩)
# -----------------------------
class TextDataset(Dataset):
    def __init__(self, xs, ys):
        self.xs = xs
        self.ys = ys
    def __len__(self): 
        return len(self.xs)
    def __getitem__(self, i): 
        return self.xs[i], self.ys[i]

MAX_K = 5  # kernel_sizes 중 최대값

def collate_fn(batch):
    xs, ys = zip(*batch)
    pad_id = stoi["<PAD>"]

    # 각 시퀀스를 최소 길이(MAX_K) 이상으로 패딩
    fixed = []
    for x in xs:
        if len(x) < MAX_K:
            need = MAX_K - len(x)
            x = torch.cat([x, torch.full((need,), pad_id, dtype=torch.long)])
        fixed.append(x)

    xs_pad = pad_sequence(fixed, batch_first=True, padding_value=pad_id)
    lengths = torch.tensor([len(x) for x in fixed], dtype=torch.long)
    return xs_pad, torch.stack(ys), lengths


train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(TextDataset(X_val,   y_val),   batch_size=8, shuffle=False, collate_fn=collate_fn)


In [4]:


# -----------------------------
# 5) 1D-CNN 모델 (Kim CNN)
#    Embedding -> Conv1d(k=3,4,5) -> ReLU -> max-over-time -> concat -> Dropout -> Linear
# -----------------------------
class TextCNN(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, num_classes: int, kernel_sizes=(3,4,5), num_channels=100, pad_idx=0, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        # Conv1d: in_channels=emb_dim, out_channels=num_channels, kernel_size=k
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=num_channels, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_channels * len(kernel_sizes), num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.emb.weight)
        for conv in self.convs:
            # 신경망 가중치(weight)를 초기화하는 함수
            # ReLU 계열 활성함수(ReLU, LeakyReLU 등)와 함께 쓸 때 학습이 안정되도록 설계된 초기화 방식
            nn.init.kaiming_uniform_(conv.weight, a=math.sqrt(5))
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):  # x: (B, T)
        # 임베딩 후 채널 차원 앞쪽으로: (B, T, E) -> (B, E, T)
        # B(배치의 크기), E(임베딩 차원의 수), T(각 문장의 단어의 개수, 시퀸스의 길이)
        x = self.emb(x)                      # (B, T, E)
        x = x.transpose(1, 2)                # (B, E, T)

        # 컨볼루션 + ReLU + max-over-time 풀링
        feat_maps = []
        for conv in self.convs:
            h = torch.relu(conv(x))          # (B, C, T')
            h = torch.max(h, dim=2).values   # (B, C)  # time dim max
            # C : relu에서 나온 차원의 개수
            # T' = T - kernel_size + 1
            feat_maps.append(h)
        # 열을 기준으로 결합
        z = torch.cat(feat_maps, dim=1)      # (B, C * #kernels)
        # nn.Dropout()은 신경망 학습에서 과적합(overfitting)을 막기 위해
        #  일부 뉴런(노드)의 출력을 임시로 “0”으로 만들어 버리는 정규화 기법
        z = self.dropout(z)
        logits = self.fc(z)                   # (B, num_classes)
        return logits


In [5]:

# -----------------------------
# 6) 학습 루프
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN(vocab_size=len(vocab), emb_dim=128, num_classes=2, kernel_sizes=(3,4,5), num_channels=64, pad_idx=stoi["<PAD>"], dropout=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=3e-3)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y, lengths in loader:
        x, y = x.to(device), y.to(device)
        # 자동 미분 활성화
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optim.zero_grad()
                loss.backward()
                optim.step()
        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=1)
        correct += int((preds == y).sum().item())
        total += x.size(0)
    return total_loss/total, correct/total

EPOCHS = 10
for ep in range(1, EPOCHS+1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    print(f"[{ep:02d}] train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")


[01] train loss 0.6699 acc 0.594 | val loss 0.5598 acc 0.690
[02] train loss 0.3224 acc 0.871 | val loss 0.6999 acc 0.653
[03] train loss 0.1157 acc 0.959 | val loss 0.8936 acc 0.660
[04] train loss 0.0890 acc 0.961 | val loss 0.9706 acc 0.663
[05] train loss 0.0648 acc 0.963 | val loss 1.0229 acc 0.690
[06] train loss 0.0563 acc 0.963 | val loss 1.1188 acc 0.703
[07] train loss 0.0617 acc 0.966 | val loss 1.2857 acc 0.677
[08] train loss 0.0591 acc 0.969 | val loss 1.2206 acc 0.687
[09] train loss 0.0549 acc 0.970 | val loss 1.1521 acc 0.677
[10] train loss 0.0549 acc 0.969 | val loss 1.2172 acc 0.657


In [6]:

# -----------------------------
# 7) 예측 함수
# -----------------------------
@torch.no_grad()
def predict(text: str):
    toks = tokenize(text)
    ids = encode(toks)                # 1D LongTensor
    pad_id = stoi["<PAD>"]
    # 모델의 최대 커널 크기 확인
    max_k = max([m.kernel_size[0] for m in model.convs])
    if len(ids) < max_k:
        need = max_k - len(ids)
        ids = torch.cat([ids, torch.full((need,), pad_id, dtype=torch.long)])
    x = ids.unsqueeze(0).to(device)   # (1, T>=max_k)
    logits = model(x)
    prob = torch.softmax(logits, dim=1).squeeze(0).cpu().tolist()
    pred = int(torch.argmax(logits, dim=1).item())
    return pred, prob

print("\n[예측 예시]")
for s in ["케익이 훌륭하고 만족스러웠다", "직원 태도가 별로였고 실망했다"]:
    yhat, p = predict(s)
    print(f"{s} -> pred={yhat} prob={list(map(lambda x: round(x,3), p))}")



[예측 예시]
케익이 훌륭하고 만족스러웠다 -> pred=1 prob=[0.0, 1.0]
직원 태도가 별로였고 실망했다 -> pred=0 prob=[1.0, 0.0]
